# Week 39, Day 6: Forge: Week 39 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w039/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
print("1.", round(24.7 * (4.37 * 4 + 1)), "Hz")
th = np.deg2rad(30); print("2.", round(0.0875 / 343 * (th + np.sin(th)) * 1e6), "microseconds")
print("3.", round(0.5 * np.log2(1 + 9), 2), "bits")

### Exercise: Part A: adaptive rescaling

Write running_sd(s, alpha=0.01), an exponentially weighted estimate of the stimulus standard deviation (v = (1 - alpha) v + alpha s², starting at 1, returning sqrt(v) at each sample). Use it as the adaptive gain. The response, information estimate and comparisons are given.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2000)
T, blk = 40000, 2000
sig = np.repeat(np.tile([1.0, 4.0], T // (2 * blk)), blk)
s = r.normal(size=T) * sig
settle = np.tile(np.r_[np.zeros(300, bool), np.ones(blk - 300, bool)], T // blk)

def respond(gain):
    x = np.tanh(s / gain) + 0.05 * r.normal(size=T)
    return np.digitize(x, np.linspace(-1, 1, 9)[1:-1])          # 8 response levels

def mutual_info(a, b, nb=16):
    qa = np.digitize(a, np.quantile(a, np.linspace(0, 1, nb + 1)[1:-1]))
    P = np.histogram2d(qa, b, bins=[nb, 8])[0]; P /= P.sum()
    px, py = P.sum(1, keepdims=True), P.sum(0, keepdims=True); nz = P > 0
    return np.sum(P[nz] * np.log2(P[nz] / (px @ py)[nz]))

def running_sd(s, alpha=0.01):
    ...

gains = {"fixed 1": 1.0, "fixed 2": 2.0, "fixed 4": 4.0, "adaptive": running_sd(s)}
within, overall = {}, {}
for name, g in gains.items():
    resp = respond(g)
    within[name] = [mutual_info(s[(sig == sv) & settle], resp[(sig == sv) & settle]) for sv in (1.0, 4.0)]
    overall[name] = mutual_info(s, resp)
    print(f"{name:9s} within low {within[name][0]:.2f}, within high {within[name][1]:.2f} bits;  overall {overall[name]:.2f} bits")

In [ ]:
import numpy as np
def test_within():
    "the adaptive gain gives the highest mean within-ensemble information"
    assert max(within, key=lambda k: np.mean(within[k])) == "adaptive"

def test_balanced():
    "each fixed gain is good in at most one ensemble; the adaptive gain is within 0.2 bits of the best in both"
    best = [max(within[k][i] for k in within) for i in range(2)]
    assert all(within["adaptive"][i] > best[i] - 0.2 for i in range(2))

def test_overall():
    "over the whole recording, ignoring which ensemble is on, the adaptive gain carries less information than the best fixed gain"
    assert overall["adaptive"] < max(overall[k] for k in overall if k != "adaptive")

test_within()
test_balanced()
test_overall()
print("All checks passed")

### Exercise: Part B: information under a power limit

Write info_bits(K2), which first rescales the squared gains K2 so that the output power equals P, stores the scaled gains, and returns the information in bits. Then evaluate the three filters.

In [ ]:
k = np.arange(1, 257)
S = 1e4 / k ** 2
P = 50.0
scaled = {}

def info_bits(name, K2):
    K2 = ...
    scaled[name] = K2
    return ...

bits = {"no filter": info_bits("no filter", np.ones_like(S)),
        "whitening": info_bits("whitening", 1 / (S + 1)),
        "Atick and Redlich": info_bits("Atick and Redlich", (S / (S + 1)) ** 2 / (S + 1))}
print({name: round(float(b), 1) for name, b in bits.items()}, "bits")
for name, K2 in scaled.items():
    plt.loglog(k, K2, label=name)
plt.xlabel("frequency index"); plt.ylabel("squared gain (scaled)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_order():
    "the Atick and Redlich filter carries the most information, then whitening, then no filter"
    assert bits["Atick and Redlich"] > bits["whitening"] > bits["no filter"]

def test_power():
    "every filter uses the same output power"
    assert all(abs(np.sum(scaled[k] * (S + 1)) - P) < 1e-6 for k in scaled)

test_order()
test_power()
print("All checks passed")